# 6_agent_tr — Gün Öncesi Karar Zinciri: Tahmin → Marj → Karar → RAG → Agent

Bu notebook, Faz 3'ün 4-5. adımlarını **geçmiş verinin yeniden oynatılmasıyla** kuruyor. Seçilen bir D günü için, tahmin anında (D-1 akşamı) bilinebilecek bilgiyle zincirin tamamı çalışıyor. Sonuç, gerçekleşen tüketimle karşılaştırılıyor. Canlı veri toplayıcı (Faz 3, sonraki adım), doğrulanmış bu zincirin önüne takılacak.

| Bölüm | İş |
|---|---|
| **A** | Kurulum, model/ayar dosyaları, veri |
| **B** | Gün öncesi tahmin: yedek modele geçiş kuralları, son 30 günlük GFS sapma düzeltmesi, temkinli tahmin (q90) |
| **C** | EAK profili (gün içi sıçrama düzeltmesi, güneş anomali kontrolü), marj ve **deterministik** karar |
| **D** | RAG: duruma göre mevzuat parçaları |
| **E** | Agent: kararı açıklama ve mevzuata dayandırma; iki güvenlik kontrolü |
| **F** | Zincir ve 6 senaryo testi |
| **G** | n8n yükü (varsayılan: gönderilmiyor) |

**Agent tasarımı:**
- **Karar LLM'den gelmiyor.** Karar, eşik tablosundan kodla üretiliyor. LLM kararı açıklıyor ve mevzuata dayandırıyor.
- **Kod iki şeyi denetliyor:**
  - LLM'in döndürdüğü durum ve kararın kuralın kararıyla aynı olması.
  - Her mevzuat atfının, gerçekten getirilen parçalar arasında olması; uydurma atıflar çıkarılıyor.
- **Model:** `gemini-3.7-flash` (kararlı sürüm).
- **Sıcaklık ayarlanmıyor.** Google, Gemini 3 modellerinde sıcaklığın varsayılan 1,0'da bırakılmasını öneriyor; düşürmek döngü veya performans düşüşüne yol açabiliyor. Tekrarlanabilir biçim için bunun yerine JSON şemalı yapılandırılmış çıktı kullanılıyor.
- **Düşünme seviyesi `low`.** Bu model `minimal` seviyesini desteklemiyor.

## A. Kurulum ve Yükleme

### A1. Kurulum

`AGENT_CAGIR = False` yapılırsa Gemini çağrılmaz; kararın açıklaması kodla üretilir. Kotayı korumak veya yalnızca zinciri test etmek için kullanılabilir.

In [1]:
import os
import sys
import json
import re
import time
import numpy as np
import pandas as pd
import xgboost as xgb
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langsmith import traceable

KOK = r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus"
load_dotenv(os.path.join(KOK, "notepads", ".env"))
os.environ["LANGSMITH_PROJECT"] = "tr-grid-agent"
sys.path.insert(0, os.path.join(KOK, "notepads"))
import tr_rag_retrieval as trr

GEMINI_MODEL = "gemini-3.7-flash"      # alternatif: "gemini-3.8-flash"
AGENT_CAGIR = True
gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Model:", GEMINI_MODEL, "| Agent cagrilacak mi:", AGENT_CAGIR)

Model: gemini-3.7-flash | Agent cagrilacak mi: True


### A2. Modeller, eşikler ve belirsizlik bandı

- **Üç gün öncesi model:** seçilen, hava tahminsiz yedek ve plansız yedek. Her biri kendi meta dosyasıyla yükleniyor; özellik listesi, hedef türü ve bazlar meta dosyasından okunuyor.
- **Eşikler:** `agent_esikleri.json`.
- **Belirsizlik bandı:** Seçilen modelin, dört çapraz doğrulama katmanından hesaplanmış saatlik q90 değerleri (`tr_gunoncesi_q90.json`). Yedek modeller için ayrı bir bant hesaplanmadı. Bu yüzden onlarda aynı bant **yaklaşık** olarak kullanılıyor ve bu durum bir uyarı olarak raporlanıyor.

In [2]:
def yukle(dosya):
    m = xgb.XGBRegressor()
    m.load_model(os.path.join(KOK, "agent", dosya + ".json"))
    with open(os.path.join(KOK, "agent", dosya + "_meta.json"), encoding="utf-8") as f:
        return {"model": m, "meta": json.load(f)}

MODELLER = {"secilen": yukle("tr_xgb_gunoncesi_secilen"),
            "havasiz": yukle("tr_xgb_gunoncesi_yedek_havasiz"),
            "plansiz": yukle("tr_xgb_gunoncesi_yedek_plansiz")}
for k, v in MODELLER.items():
    print(f"{k:8s}: {v['meta']['model']} | hedef {v['meta']['hedef']} | {len(v['meta']['ozellikler'])} ozellik")

with open(os.path.join(KOK, "metrics", "agent_esikleri.json"), encoding="utf-8") as f:
    ESIK = json.load(f)
with open(os.path.join(KOK, "metrics", "tr_gunoncesi_q90.json"), encoding="utf-8") as f:
    q90j = json.load(f)
Q90 = np.array([q90j["q90_saatlik"][str(h)] for h in range(24)])
META_S = MODELLER["secilen"]["meta"]
OM_KULLAN = "om_t" in META_S["ozellikler"]
print("\nEsikler:", {k: ESIK[k] for k in ["cok_nadir_alti", "nadir_alti"]}, "| q90 araligi: %"
      f"{Q90.min() * 100:.2f}-{Q90.max() * 100:.2f} | secilen model hava tahmini kullaniyor: {OM_KULLAN}")

secilen : Hibrit (oran) + Open-Meteo | hedef oran | 22 ozellik
havasiz : Hibrit (oran) | hedef oran | 21 ozellik
plansiz : Referans (3_model) | hedef seviye | 17 ozellik

Esikler: {'cok_nadir_alti': 0.207, 'nadir_alti': 0.246} | q90 araligi: %3.70-5.18 | secilen model hava tahmini kullaniyor: True


### A3. Veri

- **Tüketim ve takvim özellikleri:** `tr_ozellikli_saatlik.csv`.
- **Resmî plan:** Ham hâli (`lep_ham`) korunuyor. **Geçmiş** özellikler (plan hatası gecikmeleri) için `3c` ile aynı temizlik kuralları uygulanıyor. D gününün kendi planı ise yalnızca D-1'de bilinebilecek kontrollerle (eksik, sıfır, düz plan) değerlendiriliyor. Tüketime bağlı oran kuralı D günü için kullanılmıyor, çünkü D'nin tüketimi o anda bilinmiyor.
- **Hava:** `3b`/`3c`'nin önbelleğe aldığı ERA5 ve GFS gün öncesi tahmin dosyaları.
- **EAK:** Ham `tr_eak_saatlik.csv`. Gün içi sıçrama düzeltmesi C bölümünde, yalnızca D gününün profili içinde yapılıyor.

In [3]:
df = pd.read_csv(os.path.join(KOK, "data", "tr_ozellikli_saatlik.csv"))
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
lp = pd.read_csv(os.path.join(KOK, "data", "tr_yuk_plani_saatlik.csv"))
lp["date"] = pd.to_datetime(lp["date"])
df = df.merge(lp[["date", "lep"]].rename(columns={"lep": "lep_ham"}), on="date", how="left")
df["gun_t"] = df["date"].dt.tz_localize(None).dt.normalize()

# Gecmis ozellikler icin plan temizligi (3c ile ayni kurallar)
oran = df["consumption"] / df["lep_ham"]
lep = df["lep_ham"].where(~(df["lep_ham"].notna() & ((df["lep_ham"] <= 0) | (oran > 2) | (oran < 0.5))))
gstat = lep.groupby(df["gun_t"]).agg(["std", "mean", "count"])
duz_gunler = gstat.index[(gstat["count"] == 24) & (gstat["std"] < 0.01 * gstat["mean"])]
df["lep"] = lep.where(~df["gun_t"].isin(duz_gunler))
print("Gecmiste duz plan gunu:", [str(g.date()) for g in duz_gunler])

df["lag_48"] = df["consumption"].shift(48)
df["lag_72"] = df["consumption"].shift(72)
df["rol_24_48"] = df["consumption"].shift(48).rolling(24).mean()
df["rol_168_48"] = df["consumption"].shift(48).rolling(168).mean()
df["tavg_dun"] = df["tavg_c"].shift(24)
df["lep_hata"] = df["lep"] - df["consumption"]
df["lep_hata_48"] = df["lep_hata"].shift(48)
df["lep_hata_168"] = df["lep_hata"].shift(168)
df["lep_hata_ort_7g"] = df["lep_hata"].shift(48).rolling(168, min_periods=120).mean()

# Hava (yalnizca secilen model hava tahmini kullaniyorsa)
if OM_KULLAN:
    agirlik = META_S["sehir_agirliklari"]
    def seri(dosya):
        s = pd.read_csv(os.path.join(KOK, "data", dosya))
        return pd.Series(s["deger"].values, index=pd.to_datetime(s["date"]))
    idx = pd.DatetimeIndex(df["date"])
    era5 = pd.DataFrame({s: seri(f"om_era5_{s}.csv") for s in agirlik})
    tahm = pd.DataFrame({s: seri(f"om_tahmin_uzun_{s}.csv") for s in agirlik})
    df["om_era5"] = (era5 * pd.Series(agirlik)).sum(axis=1, min_count=len(agirlik)).reindex(idx).values
    df["om_fc_ham"] = (tahm * pd.Series(agirlik)).sum(axis=1, min_count=len(agirlik)).reindex(idx).values

eak = pd.read_csv(os.path.join(KOK, "data", "tr_eak_saatlik.csv"))
eak["date"] = pd.to_datetime(eak["date"])
df = df.merge(eak[["date", "toplam", "gunes"]].rename(columns={"toplam": "eak_toplam", "gunes": "eak_gunes"}),
              on="date", how="left")
print("Veri:", len(df), "saat |", df["date"].min(), "->", df["date"].max())

Gecmiste duz plan gunu: ['2025-04-07']
Veri: 91968 saat | 2016-04-01 00:00:00+03:00 -> 2026-09-27 23:00:00+03:00


## B. Gün Öncesi Tahmin

**Model seçimi (D-1'de bilinen bilgiyle):**

| Durum | Model |
|---|---|
| Resmî plan tam (24 saat, sıfırdan büyük, düz değil) ve hava tahmini mevcut | Seçilen model |
| Plan tam, hava tahmini eksik | Hava tahminsiz yedek (Hibrit, oran) |
| Plan eksik, sıfır veya düz | Plansız yedek (Referans) |

**GFS sapma düzeltmesi:**
- D gününden **8 gün önce** biten 30 günlük pencerede, her saat için (tahmin − ERA5) ortalaması hesaplanıyor. 8 günlük boşluk, ERA5'in birkaç gün gecikmeyle yayımlanmasına karşı bir pay.
- Bir saatte 10'dan az gözlem varsa, meta dosyasındaki ay × saat tablosu kullanılıyor.

**Temkinli tahmin:** tahmin × (1 + q90 [saat]).

**Özel gün uyarısı:** D günü bayram, tatil veya idari izin günüyse uyarı veriliyor. Gerekçe, `3c`'nin test yılı gözlemi: hibrit model ve resmî plan bu günlerde daha yüksek hata yaptı.

In [4]:
def sapma_saatlik(D):
    gun = D.tz_localize(None).normalize() if D.tzinfo else D.normalize()   # saat dilimsiz gun uzerinden karsilastirma
    bitis = gun - pd.Timedelta(days=8)
    bas = bitis - pd.Timedelta(days=30)
    m = (df["gun_t"] >= bas) & (df["gun_t"] < bitis) & df["om_fc_ham"].notna() & df["om_era5"].notna()
    fk = (df.loc[m, "om_fc_ham"] - df.loc[m, "om_era5"]).groupby(df.loc[m, "date"].dt.hour)
    ort, say = fk.mean(), fk.size()
    tablo = META_S.get("sapma_tablosu_ay_saat_c", {})
    sap, kaynak = np.zeros(24), []
    for h in range(24):
        if say.get(h, 0) >= 10:
            sap[h] = ort[h]; kaynak.append("30 gun")
        else:
            sap[h] = tablo.get(f"{D.month}-{h}", 0.0); kaynak.append("meta tablo")
    return sap, max(set(kaynak), key=kaynak.count)

def gun_oncesi_tahmin(D):
    g = df[df["gun_t"] == D.tz_localize(None).normalize()].copy() if D.tzinfo else df[df["gun_t"] == D].copy()
    if len(g) != 24:
        raise ValueError(f"{D.date()} icin 24 saatlik veri yok ({len(g)})")
    uyarilar = []
    lp_d = g["lep_ham"]
    plan_ok = bool(lp_d.notna().all() and (lp_d > 0).all() and lp_d.std() >= 0.01 * lp_d.mean())
    if not plan_ok:
        uyarilar.append("Resmi plan eksik veya supheli (eksik, sifir veya duz); plansiz yedek model kullanildi")
    om_ok = OM_KULLAN and bool(g["om_fc_ham"].notna().all())
    if OM_KULLAN and not om_ok:
        uyarilar.append("Gun oncesi hava tahmini eksik; hava tahminsiz yedek model kullanildi")
    anahtar = "secilen" if plan_ok and (om_ok or not OM_KULLAN) else ("havasiz" if plan_ok else "plansiz")
    meta, model = MODELLER[anahtar]["meta"], MODELLER[anahtar]["model"]
    if anahtar != "secilen":
        uyarilar.append("Belirsizlik bandi secilen modelden alindi; yedek model icin yaklasik")

    bh, bc = meta["ankara_bazlari_c"]
    g["hdd_dun"] = np.maximum(0, bh - g["tavg_dun"])
    g["cdd_dun"] = np.maximum(0, g["tavg_dun"] - bc)
    sapma_kaynak = None
    if "om_t" in meta["ozellikler"]:
        sap, sapma_kaynak = sapma_saatlik(D)
        g["om_t"] = g["om_fc_ham"].values - sap[g["saat"].values]
        g["om_tgun"] = g["om_t"].mean()
        obh, obc = meta["om_bazlari_c"]
        g["om_hdd"] = np.maximum(0, obh - g["om_tgun"])
        g["om_cdd"] = np.maximum(0, g["om_tgun"] - obc)
    g["lep"] = g["lep_ham"]                     # D gununun plani: yalnizca D-1'de bilinen kontrollerle
    X = g[meta["ozellikler"]]
    if X.isna().any().any():
        uyarilar.append(f"Model girdisinde {int(X.isna().sum().sum())} eksik deger (XGBoost eksik degeri kendi isler)")
    p = model.predict(X)
    tahmin = g["lep"].values * (1 + p) if meta["hedef"] == "oran" else p
    temkinli = tahmin * (1 + Q90[g["saat"].values])

    ozel = []
    if (g["bayram_once_gun"] == 0).all() and (g["bayram_sonra_gun"] == 0).all():
        ozel.append("bayram veya arife")
    if (g["tatil_tam"] == 1).any() and not ozel:
        ozel.append("resmi tatil")
    if (g["idari_izin"] == 1).any():
        ozel.append("idari izin")
    if ozel:
        uyarilar.append(f"Ozel gun ({', '.join(ozel)}): test yilinda resmi plan ve hibrit model bu gunlerde daha yuksek hata yapti")
    return {"g": g, "tahmin": np.asarray(tahmin, float), "temkinli": np.asarray(temkinli, float),
            "model_anahtar": anahtar, "model": meta["model"], "sapma_kaynak": sapma_kaynak, "uyarilar": uyarilar}

## C. EAK Profili, Marj ve Deterministik Karar

**EAK kontrolleri (yalnızca D günü ve öncesiyle):**
- **Sıçrama:** Profil içinde, bir saatte büyük artış ve hemen ardından büyük düşüş (ya da tersi) aranıyor. Eşik, önceki 365 günün saatlik değişimlerinin 4 standart sapması (Faz 1'deki kural). Bulunan saatler komşu saatlerden doğrusal olarak dolduruluyor.
- **Güneş anomalisi:** Öğlen güneş EAK'sı, referansın yarısından azsa uyarı veriliyor. Referans, iki medyanın büyüğü: son 14 günün öğlen medyanı ve D-60 ile D-15 arasının öğlen medyanı. Uzun referans, **uzun süren** anomalileri yakalamak için var. Faz 1'deki Mayıs 2026 anomalisi 21 gün sürdü; yalnızca son 14 günle bakılsaydı, anomalinin ikinci haftasında referans da anomalili olurdu ve düşüş görülmezdi.

**Karar tablosu (`agent_esikleri.json`):**

| Koşul | Durum | Karar | Öncelik |
|---|---|---|---|
| Temkinli tahminle en düşük marj ≤ 0 | action_required | `activate_reserve` | yüksek |
| Nokta tahminle en düşük marj < çok nadir eşiği | attention | `demand_response_alert` | yüksek |
| Nokta tahminle en düşük marj < nadir eşiği | attention | `demand_response_alert` | normal |
| Diğer | normal | `normal_operation` | yok |

Marj = (EAK − tahmin) / EAK.

In [5]:
def eak_profili(D, carpan=1.0):
    gun = D.tz_localize(None).normalize() if D.tzinfo else D
    g = df[df["gun_t"] == gun]
    uyarilar = []
    if g["eak_toplam"].isna().any() or len(g) != 24:
        raise ValueError(f"{gun.date()} icin EAK eksik")
    t = g["eak_toplam"].reset_index(drop=True) * carpan
    once = df[(df["gun_t"] < gun) & (df["gun_t"] >= gun - pd.Timedelta(days=365))]["eak_toplam"]
    esik = 4 * once.diff().abs().std()
    fark = t.diff()
    sicrama = ((fark > esik) & (fark.shift(-1) < -esik)) | ((fark < -esik) & (fark.shift(-1) > esik))
    t_duz = t.mask(sicrama).interpolate(limit_direction="both")
    if sicrama.any():
        uyarilar.append(f"EAK profilinde {int(sicrama.sum())} saatlik sicrama duzeltildi (saat: {[int(x) for x in np.where(sicrama)[0]]})")
    oglen = g.loc[g["saat"] == 12, "eak_gunes"]
    o12 = df[df["saat"] == 12]
    ref_kisa = o12[(o12["gun_t"] < gun) & (o12["gun_t"] >= gun - pd.Timedelta(days=14))]["eak_gunes"].median()
    ref_uzun = o12[(o12["gun_t"] < gun - pd.Timedelta(days=14)) & (o12["gun_t"] >= gun - pd.Timedelta(days=60))]["eak_gunes"].median()
    ref = np.nanmax([ref_kisa, ref_uzun]) if pd.notna(ref_kisa) or pd.notna(ref_uzun) else np.nan
    if len(oglen) and pd.notna(ref) and ref > 1000 and pd.notna(oglen.iloc[0]) and oglen.iloc[0] < 0.5 * ref:
        uyarilar.append(f"EAK gunes bileseninde ani dusus: oglen {oglen.iloc[0]:.0f} MWh; referans {ref:.0f} MWh "
                        f"(son 14 gun medyani {ref_kisa:.0f}, D-60..D-15 medyani {ref_uzun:.0f})")
    if carpan != 1.0:
        uyarilar.append(f"YAPAY SENARYO: EAK x {carpan:.4f} (EAK kaybi %{(1 - carpan) * 100:.1f})")
    return {"eak": t_duz.values.astype(float), "uyarilar": uyarilar}

def karar_ver(tahmin, temkinli, eak_v):
    marj = (eak_v - tahmin) / eak_v
    marj_t = (eak_v - temkinli) / eak_v
    i, it = int(np.argmin(marj)), int(np.argmin(marj_t))
    if marj_t.min() <= 0:
        durum, karar, oncelik = "action_required", "activate_reserve", "yuksek"
    elif marj.min() < ESIK["cok_nadir_alti"]:
        durum, karar, oncelik = "attention", "demand_response_alert", "yuksek"
    elif marj.min() < ESIK["nadir_alti"]:
        durum, karar, oncelik = "attention", "demand_response_alert", "normal"
    else:
        durum, karar, oncelik = "normal", "normal_operation", "yok"
    return {"durum": durum, "karar": karar, "oncelik": oncelik, "en_dar_saat": i,
            "en_dusuk_marj": round(float(marj.min()), 4), "en_dar_saat_temkinli": it,
            "en_dusuk_marj_temkinli": round(float(marj_t.min()), 4), "marj": marj, "marj_t": marj_t}
print("C hazir.")

C hazir.


## D. RAG

Karar durumuna göre `build_rag_query_tr` sorgusu ile 4 parça getiriliyor. Veri uyarısı varsa, veri yayın zamanlarıyla ilgili 2 parça daha ekleniyor. Getirilen maddelerin ilk 250 karakteri yazdırılıyor; böylece getirilen metinlerin gerçekten ilgili olup olmadığı nitel olarak da kontrol edilebiliyor.

In [6]:
def mevzuat_getir(durum, veri_uyarisi):
    parcalar = trr.getir(trr.build_rag_query_tr(durum), k=4)
    if veri_uyarisi:
        parcalar += trr.getir(trr.build_rag_query_tr("veri"), k=2)
    tekil, gorulen = [], set()
    for p in parcalar:
        if p["id"] not in gorulen:
            tekil.append(p); gorulen.add(p["id"])
    return tekil

ornek = mevzuat_getir("action_required", False)
for p in ornek[:2]:
    print(f"[{p['belge']} | {p['madde']}] {p['benzerlik']}\n  {p['metin'][:250].replace(chr(10), ' ')}\n")

[Elektrik Şebeke Yönetmeliği | MADDE 67] 0.7617
  [Elektrik Şebeke Yönetmeliği | MADDE 67] MADDE 67- (Değişik: RG-17/12/2024-32755)  (1) Düşük frekans röleleri ile talebin otomatik olarak zorunlu kesilmesi, kısa  dönemli bir arz talep dengesizliğini ortadan kaldırmak amacıyla yapılır. Talebin otomat

[Elektrik Şebeke Yönetmeliği | MADDE 62] 0.7536
  [Elektrik Şebeke Yönetmeliği | MADDE 62 (parca 2/2)] şağıda belirtilen esaslar çerçevesinde acil durum önlemleri uygulanır:  a) Bu yönetmeliğin ilgili maddesi kapsamında üretim faaliyeti gösteren tüzel  kişilere acil durum bildirimi yapılması,  b) El



## E. Agent

**Girdi:** Kodun ürettiği karar özeti (`<karar_ozeti>`) ve getirilen mevzuat parçaları (`<mevzuat_baglami>`). PJM projesindeki XML etiketli yapı; girdiyi talimattan ayırıyor.

**Çıktı:** JSON şemasıyla zorunlu kılınmış yapı: durum, karar, öncelik, özet, gerekçeler, mevzuat atıfları, veri uyarıları, belirsizlik notu.

**Kodla doğrulama:**
1. Durum, karar ve öncelik kuralın ürettiğiyle aynı değilse düzeltilip not düşülüyor.
2. Getirilen parçalar arasında olmayan bir atıf çıkarılıp not düşülüyor. Atıf, belge adı getirilen bir parçayla aynıysa **ve** madde etiketi o parçanın etiketiyle aynıysa ya da madde başlığı o parçanın metninde geçiyorsa geçerli sayılıyor. İkinci koşul, bir ek maddenin (örneğin MADDE 67/A) bir önceki maddenin parçası içinde kalması durumunu karşılıyor. Eşleştirme kelime sınırlı yapılıyor: "MADDE 4", "MADDE 40"ı; "MADDE 67", "MADDE 67/A"yı eşlemiyor.

**Hata durumu:** Gemini çağrısı başarısız olursa (kota vb.), açıklama kodla üretiliyor ve zincir kesilmiyor.

In [7]:
SISTEM = """Sen Türkiye elektrik sistemi için bir karar destek asistanısın. Görevin, kural tabanlı olarak üretilmiş gün öncesi kararı açıklamak ve mevzuata dayandırmaktır.
Kurallar:
1. <karar_ozeti> içindeki durum, karar ve öncelik kesindir; değiştirme. Çıktındaki bu alanlar aynen bunlar olmalı.
2. Mevzuat atıflarını YALNIZCA <mevzuat_baglami> içindeki parçalardan yap. Her atıfta parçanın köşeli parantez içindeki başlığındaki belge adını ve madde etiketini aynen yaz. Bağlamda olmayan bir madde numarası yazma. Uygun parça yoksa atıf listesini boş bırak.
3. Eşikler tarihsel nadirlik eşikleridir (Mart 2025 sonrası marj dağılımının %1 ve %5'lik dilimleri); fiziksel bir güvenilirlik sınırı değildir. Bunu gerekçede belirt.
4. Veri uyarılarını veri_uyarilari alanına aynen aktar.
5. Kısa, teknik ve Türkçe yaz."""

SEMA = {"type": "object", "properties": {
    "durum": {"type": "string", "enum": ["normal", "attention", "action_required"]},
    "karar": {"type": "string", "enum": ["normal_operation", "demand_response_alert", "activate_reserve"]},
    "oncelik": {"type": "string", "enum": ["yok", "normal", "yuksek"]},
    "ozet": {"type": "string"},
    "gerekceler": {"type": "array", "items": {"type": "string"}},
    "mevzuat_atiflari": {"type": "array", "items": {"type": "object", "properties": {
        "belge": {"type": "string"}, "madde": {"type": "string"}, "neden": {"type": "string"}},
        "required": ["belge", "madde", "neden"]}},
    "veri_uyarilari": {"type": "array", "items": {"type": "string"}},
    "belirsizlik_notu": {"type": "string"}},
    "required": ["durum", "karar", "oncelik", "ozet", "gerekceler", "mevzuat_atiflari", "veri_uyarilari", "belirsizlik_notu"]}

@traceable(name="tr_agent_karar_aciklamasi")
def agent_cagir(ozet, parcalar):
    icerik = (f"<karar_ozeti>\n{json.dumps(ozet, ensure_ascii=False, indent=1)}\n</karar_ozeti>\n"
              f"<mevzuat_baglami>\n{trr.baglam_metni(parcalar)}\n</mevzuat_baglami>")
    r = gemini.models.generate_content(
        model=GEMINI_MODEL, contents=icerik,
        config=types.GenerateContentConfig(
            system_instruction=SISTEM, response_mime_type="application/json", response_json_schema=SEMA,
            thinking_config=types.ThinkingConfig(thinking_level=types.ThinkingLevel.LOW),
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)))   # arac kullanilmiyor
    return json.loads(r.text)

def kodla_aciklama(ozet, parcalar):
    return {"durum": ozet["durum"], "karar": ozet["karar"], "oncelik": ozet["oncelik"],
            "ozet": f"En dusuk marj %{ozet['en_dusuk_marj'] * 100:.1f} (saat {ozet['en_dar_saat']:02d}:00); karar {ozet['karar']}.",
            "gerekceler": [f"Esikler: cok nadir < %{ozet['esikler']['cok_nadir_alti'] * 100:.1f}, nadir < %{ozet['esikler']['nadir_alti'] * 100:.1f} (tarihsel nadirlik esikleri)."],
            "mevzuat_atiflari": [{"belge": p["belge"], "madde": p["madde"], "neden": "Durum sorgusuyla getirilen ilgili madde (LLM cagrilmadi)"} for p in parcalar[:2]],
            "veri_uyarilari": ozet["uyarilar"], "belirsizlik_notu": "Aciklama kodla uretildi."}

def _n(s):
    return " ".join(str(s).lower().split())

def dogrula(cikti, ozet, parcalar):
    notlar = []
    for k in ["durum", "karar", "oncelik"]:
        if cikti.get(k) != ozet[k]:
            notlar.append(f"LLM {k} alanini degistirmeye calisti ({cikti.get(k)} -> {ozet[k]}); kural karari uygulandi")
            cikti[k] = ozet[k]
    temiz = []
    for a in cikti.get("mevzuat_atiflari", []):
        b, m = _n(a.get("belge", "")), _n(a.get("madde", ""))
        desen = re.compile(r"(?<!\w)" + re.escape(m) + r"(?![\w/])")   # "madde 4" "madde 40"i, "madde 67" "madde 67/a"yi eslemez
        if any(b == _n(p["belge"]) and (m == _n(p["madde"]) or desen.search(_n(p["metin"]))) for p in parcalar):
            temiz.append(a)
        else:
            notlar.append(f"Getirilen parcalarda olmayan atif cikarildi: {a.get('belge')} | {a.get('madde')}")
    cikti["mevzuat_atiflari"] = temiz
    return cikti, notlar
print("E hazir.")

E hazir.


## F. Zincir ve Senaryo Testleri

`zincir(D)` bütün adımları çalıştırıyor ve sonucu, gerçekleşen tüketim ve resmî planla karşılaştırıyor.

| # | Gün | Neden | Beklenen |
|---|---|---|---|
| 1 | 28 Temmuz 2025 | Yeni dönemin en dar saati (20:00, marj %14,6); tüketim rekoru 14:00'te | `demand_response_alert`; en dar saat akşam |
| 2 | 7 Kasım 2025 | Yaz dışı akşam darlığı (18:00, %16,7) | `demand_response_alert` |
| 3 | 25 Mayıs 2026 | İdari izin; resmî planın hatası %13,9 | Özel gün uyarısı |
| 4 | 7 Nisan 2025 | Resmî plan neredeyse düz (veri hatası) | Plansız yedek model ve plan uyarısı |
| 5 | 1 Mayıs 2026 | EAK güneş anomalisinin ilk günü. Faz 1'e göre öğlen güneş EAK'sı 6.905 MWh, 24-30 Nisan'da 15.347-18.870 MWh. Gün aynı zamanda resmî tatil (1 Mayıs), bu yüzden özel gün uyarısı da beklenir | EAK güneş uyarısı |
| 6 | 28 Temmuz 2025, yapay kapasite açığı | EAK, en kritik saatte temkinli tahminin %1 altına inecek şekilde bir çarpanla küçültülüyor. Çarpan, o gün için "rezerv kararını tetikleyecek EAK kaybını" da gösteriyor | `activate_reserve` |

1 ve 2 numaralı senaryolarda beklenen karar, **gerçekleşen** tüketimin ürettiği marja dayanıyor. Tahmin, gerçekleşenden farklı bir marj üretebilir. Bu yüzden tabloda gerçekleşen tüketimle hesaplanan marj da gösteriliyor.

In [8]:
def zincir(gun, eak_carpan=1.0, agent=None):
    agent = AGENT_CAGIR if agent is None else agent
    D = pd.Timestamp(gun)
    t = gun_oncesi_tahmin(D)
    if eak_carpan == "acik":      # en kritik saatte EAK'yi temkinli tahminin %1 altina indiren carpan
        e0 = eak_profili(D, 1.0)
        eak_carpan = 0.99 * float(np.max(t["temkinli"] / e0["eak"]))
    e = eak_profili(D, eak_carpan)
    k = karar_ver(t["tahmin"], t["temkinli"], e["eak"])
    uyarilar = t["uyarilar"] + e["uyarilar"]
    parcalar = mevzuat_getir(k["durum"], bool(uyarilar))
    ozet = {"gun": str(D.date()), "durum": k["durum"], "karar": k["karar"], "oncelik": k["oncelik"],
            "en_dar_saat": k["en_dar_saat"], "en_dusuk_marj": k["en_dusuk_marj"],
            "en_dar_saat_temkinli": k["en_dar_saat_temkinli"], "en_dusuk_marj_temkinli": k["en_dusuk_marj_temkinli"],
            "esikler": {"cok_nadir_alti": ESIK["cok_nadir_alti"], "nadir_alti": ESIK["nadir_alti"]},
            "tahmin_zirve_mwh": round(float(t["tahmin"].max())), "eak_en_dar_saatte_mwh": round(float(e["eak"][k["en_dar_saat"]])),
            "model": t["model"], "sapma_kaynagi": t["sapma_kaynak"], "uyarilar": uyarilar}
    llm_notu = ""
    if agent:
        try:
            cikti = agent_cagir(ozet, parcalar)
        except Exception as ex:
            llm_notu = f"Gemini cagrisi basarisiz ({str(ex)[:100]}); aciklama kodla uretildi"
            cikti = kodla_aciklama(ozet, parcalar)
    else:
        cikti = kodla_aciklama(ozet, parcalar)
    cikti, notlar = dogrula(cikti, ozet, parcalar)
    if llm_notu:
        notlar.append(llm_notu)
    g = t["g"]
    ger = g["consumption"].values
    marj_ger = (e["eak"] - ger) / e["eak"]
    kiyas = {"gun_mape_%": round(float(np.mean(np.abs(t["tahmin"] - ger) / ger) * 100), 2),
             "resmi_plan_mape_%": round(float(np.nanmean(np.abs(g["lep_ham"].values - ger) / ger) * 100), 2) if g["lep_ham"].notna().any() else None,
             "gercek_en_dar_saat": int(np.argmin(marj_ger)), "gercek_en_dusuk_marj": round(float(marj_ger.min()), 4)}
    return {"ozet": ozet, "cikti": cikti, "notlar": notlar, "parcalar": parcalar, "kiyas": kiyas, "t": t, "k": k}

SENARYOLAR = [
    ("1 Yaz aksam darligi", "2025-07-28", 1.0, {"karar": "demand_response_alert"}),
    ("2 Kasim aksam darligi", "2025-11-07", 1.0, {"karar": "demand_response_alert"}),
    ("3 Idari izin gunu", "2026-05-25", 1.0, {"uyari": "Ozel gun"}),
    ("4 Duz resmi plan", "2025-04-07", 1.0, {"model_anahtar": "plansiz", "uyari": "Resmi plan"}),
    ("5 EAK gunes anomalisi", "2026-05-01", 1.0, {"uyari": "EAK gunes"}),
    ("6 Yapay kapasite acigi", "2025-07-28", "acik", {"karar": "activate_reserve"}),
]
satirlar, ciktilar = [], {}
for ad, gun, carpan, beklenen in SENARYOLAR:
    t0 = time.time()
    try:
        r = zincir(gun, carpan)
    except Exception as ex:
        satirlar.append({"senaryo": ad, "gun": gun, "gecti": False, "hata": str(ex)[:120]}); continue
    o, c = r["ozet"], r["cikti"]
    ok = True
    if "karar" in beklenen:
        ok &= c["karar"] == beklenen["karar"]
    if "model_anahtar" in beklenen:
        ok &= r["t"]["model_anahtar"] == beklenen["model_anahtar"]
    if "uyari" in beklenen:
        ok &= any(beklenen["uyari"].lower() in u.lower() for u in o["uyarilar"])
    satirlar.append({"senaryo": ad, "gun": gun, "karar": c["karar"], "oncelik": c["oncelik"],
                     "en_dar_saat": o["en_dar_saat"], "marj_%": round(o["en_dusuk_marj"] * 100, 1),
                     "marj_temkinli_%": round(o["en_dusuk_marj_temkinli"] * 100, 1),
                     "gercek_en_dar_saat": r["kiyas"]["gercek_en_dar_saat"], "gercek_marj_%": round(r["kiyas"]["gercek_en_dusuk_marj"] * 100, 1),
                     "gun_mape_%": r["kiyas"]["gun_mape_%"], "plan_mape_%": r["kiyas"]["resmi_plan_mape_%"],
                     "model": r["t"]["model_anahtar"], "atif": len(c["mevzuat_atiflari"]), "dogrulama_notu": len(r["notlar"]),
                     "gecti": bool(ok), "sure_sn": round(time.time() - t0, 1)})
    ciktilar[ad] = {"ozet": o, "cikti": c, "notlar": r["notlar"], "kiyas": r["kiyas"],
                    "getirilen": [f"{p['belge']} | {p['madde']}" for p in r["parcalar"]]}
tablo = pd.DataFrame(satirlar)
print(tablo.to_string(index=False))
print(f"\nGecen senaryo: {int(tablo['gecti'].sum())}/{len(tablo)}")
for ad, v in ciktilar.items():
    ek = [u for u in v["ozet"]["uyarilar"] if "YAPAY" in u]
    if ek:
        print(f"{ad}: {ek[0]} -> bu gun icin rezerv kararini tetikleyen EAK kaybi")
    if v["notlar"]:
        print(f"{ad} dogrulama notlari: {v['notlar']}")

               senaryo        gun                 karar oncelik  en_dar_saat  marj_%  marj_temkinli_%  gercek_en_dar_saat  gercek_marj_%  gun_mape_%  plan_mape_%   model  atif  dogrulama_notu  gecti  sure_sn
   1 Yaz aksam darligi 2025-07-28 demand_response_alert  yuksek           20     8.7              4.8                  20           14.6        4.15         1.55 secilen     3               0   True      9.5
 2 Kasim aksam darligi 2025-11-07 demand_response_alert  yuksek           18    17.7             14.0                  18           16.7        1.42         1.59 secilen     3               0   True     10.2
     3 Idari izin gunu 2026-05-25      normal_operation     yok           20    32.3             29.5                  20           37.2        8.30        13.91 secilen     2               0   True     10.6
      4 Duz resmi plan 2025-04-07      normal_operation     yok           20    36.6             33.9                  20           36.4        2.34        50.15 plansi

### F2. Bir senaryonun tam çıktısı

Senaryo 1'in Agent çıktısı, doğrulama notları ve getirilen maddeler.

In [9]:
ad = SENARYOLAR[0][0]
if ad in ciktilar:
    print(json.dumps(ciktilar[ad]["cikti"], ensure_ascii=False, indent=2))
    print("\nDogrulama notlari:", ciktilar[ad]["notlar"] or "yok")
    print("Getirilen parcalar:", ciktilar[ad]["getirilen"])
tablo.to_csv(os.path.join(KOK, "metrics", "tr_agent_senaryolari.csv"), index=False)
with open(os.path.join(KOK, "metrics", "tr_agent_ciktilari.json"), "w", encoding="utf-8") as f:
    json.dump(ciktilar, f, ensure_ascii=False, indent=2, default=str)
print("\nKaydedildi: metrics/tr_agent_senaryolari.csv, metrics/tr_agent_ciktilari.json")

{
  "durum": "attention",
  "karar": "demand_response_alert",
  "oncelik": "yuksek",
  "ozet": "2025-07-28 günü saat 20:00 için hesaplanan en düşük marj %8.66 (temkinli %4.84) ile tarihsel çok nadir eşiğinin (%20.70) oldukça altında gerçekleşmiştir. Sistemde talep tarafı katılımı uyarısı verilmesi ve dengeleme süreçlerinin izlenmesi gerekmektedir.",
  "gerekceler": [
    "Saat 20:00 itibarıyla tahmin edilen zirve talep 64.109 MWh, EAK ise 64.465 MWh olup marj %8.66 seviyesine gerilemektedir.",
    "Temkinli marj %4.84 olarak öngörülmüş ve kritik seviyelerdedir.",
    "Kullanılan eşikler (çok nadir altı: %20.70, nadir altı: %24.60) fiziksel bir güvenilirlik sınırı olmayıp, Mart 2025 sonrası marj dağılımının %1 ve %5'lik tarihsel nadirlik dilimleridir."
  ],
  "mevzuat_atiflari": [
    {
      "belge": "Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği",
      "madde": "MADDE 4",
      "neden": "Tüketim azaltma veya üretim artışına yönelik yük alma teklifleri ve piyasa tanımları kaps

### F3. Senaryo 1'in saatlik tanısı

28 Temmuz 2025'te modelin günlük hatası (%4,15), resmî planınkinden (%1,55) belirgin şekilde yüksek çıktı. Bu hücre, farkın hangi saatlerde oluştuğunu ve hava tahmininin o gün ne kadar saptığını gösteriyor. Model yeniden eğitilmiyor; yalnızca tanı amaçlı.

In [10]:
r1 = zincir("2025-07-28", agent=False)
g1 = r1["t"]["g"].reset_index(drop=True)
tani = pd.DataFrame({"saat": g1["saat"], "gercek": g1["consumption"].round(0), "resmi_plan": g1["lep_ham"].round(0),
                     "model": np.round(r1["t"]["tahmin"]), "model_hata_%": ((r1["t"]["tahmin"] - g1["consumption"]) / g1["consumption"] * 100).round(2),
                     "plan_hata_%": ((g1["lep_ham"] - g1["consumption"]) / g1["consumption"] * 100).round(2)})
print(tani.iloc[::2].to_string(index=False))
if "om_t" in g1:
    print(f"\nAgirlikli sicaklik, gunluk ortalama: GFS ham {g1['om_fc_ham'].mean():.2f} C | sapmasi duzeltilmis {g1['om_t'].mean():.2f} C | "
          f"ERA5 gerceklesen {g1['om_era5'].mean():.2f} C | sapma kaynagi: {r1['t']['sapma_kaynak']}")
print("Modelin gunluk ortalama sapmasi: %", round(float(tani["model_hata_%"].mean()), 2),
      "| resmi planin: %", round(float(tani["plan_hata_%"].mean()), 2))

 saat  gercek  resmi_plan   model  model_hata_%  plan_hata_%
    0 45663.0     47468.0 47386.0          3.77         3.95
    2 41972.0     43404.0 43162.0          2.84         3.41
    4 39850.0     41067.0 40899.0          2.63         3.05
    6 38184.0     38743.0 38366.0          0.48         1.47
    8 48165.0     48706.0 48927.0          1.58         1.12
   10 55353.0     54825.0 56593.0          2.24        -0.95
   12 56630.0     57400.0 59625.0          5.29         1.36
   14 59504.0     60026.0 64109.0          7.74         0.88
   16 59131.0     59106.0 62864.0          6.31        -0.04
   18 55729.0     55701.0 57991.0          4.06        -0.05
   20 55039.0     56289.0 58884.0          6.99         2.27
   22 53571.0     54352.0 56630.0          5.71         1.46

Agirlikli sicaklik, gunluk ortalama: GFS ham 30.80 C | sapmasi duzeltilmis 30.71 C | ERA5 gerceklesen 29.67 C | sapma kaynagi: 30 gun
Modelin gunluk ortalama sapmasi: % 4.14 | resmi planin: % 1.43


## G. n8n Yükü

PJM projesindeki webhook yapısına uygun bir yük hazırlanıyor. Türkiye için ayrı bir workflow Faz 3'ün 6. adımında kurulacak. Bu yüzden `GONDER = False` varsayılan. Adres `.env`'deki `N8N_TR_WEBHOOK_URL`'den okunuyor.

In [11]:
GONDER = False
ad = SENARYOLAR[0][0]
if ad in ciktilar:
    o, c = ciktilar[ad]["ozet"], ciktilar[ad]["cikti"]
    yuk = {"region": "TR", "gun": o["gun"], "status": c["durum"], "decision": c["karar"], "priority": c["oncelik"],
           "tightest_hour": o["en_dar_saat"], "min_margin": o["en_dusuk_marj"], "min_margin_conservative": o["en_dusuk_marj_temkinli"],
           "summary": c["ozet"], "reasons": c["gerekceler"], "citations": c["mevzuat_atiflari"],
           "data_warnings": c["veri_uyarilari"], "model": o["model"]}
    print(json.dumps(yuk, ensure_ascii=False, indent=1)[:1500])
    url = os.getenv("N8N_TR_WEBHOOK_URL")
    if GONDER and url:
        import requests
        r = requests.post(url, json=yuk, timeout=30)
        print("n8n yaniti:", r.status_code)
    else:
        print("\nGonderilmedi (GONDER = False veya N8N_TR_WEBHOOK_URL tanimli degil).")

{
 "region": "TR",
 "gun": "2025-07-28",
 "status": "attention",
 "decision": "demand_response_alert",
 "priority": "yuksek",
 "tightest_hour": 20,
 "min_margin": 0.0866,
 "min_margin_conservative": 0.0484,
 "summary": "2025-07-28 günü saat 20:00 için hesaplanan en düşük marj %8.66 (temkinli %4.84) ile tarihsel çok nadir eşiğinin (%20.70) oldukça altında gerçekleşmiştir. Sistemde talep tarafı katılımı uyarısı verilmesi ve dengeleme süreçlerinin izlenmesi gerekmektedir.",
 "reasons": [
  "Saat 20:00 itibarıyla tahmin edilen zirve talep 64.109 MWh, EAK ise 64.465 MWh olup marj %8.66 seviyesine gerilemektedir.",
  "Temkinli marj %4.84 olarak öngörülmüş ve kritik seviyelerdedir.",
  "Kullanılan eşikler (çok nadir altı: %20.70, nadir altı: %24.60) fiziksel bir güvenilirlik sınırı olmayıp, Mart 2025 sonrası marj dağılımının %1 ve %5'lik tarihsel nadirlik dilimleridir."
 ],
 "citations": [
  {
   "belge": "Elektrik Piyasası Dengeleme ve Uzlaştırma Yönetmeliği",
   "madde": "MADDE 4",
   "nede